## Chapter 02

## INTRODUCTION TO PYTHON LIBRARY Plotly

---

Firstly, since you are seeing them from GitHub most probably, there are rendering issues with `Plotly` inside GitHub. So, we are using a different method, this entire Jupyter notebook will be turned into a live page automatically. For that we need the following lines of codes and the other set up is done inside GitHub path "*.github / workflows / build.yml*"

In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In JupyterLab, modern Plotly (>= 5.0) renders interactive WebGL canvases natively without needing extra Jupyter extensions. The core building block for full programmatic control is plotly.graph_objects (conventionally imported as go), where every figure consists of two parts:
1. *data* : A list of trace objects (like go.Surface).
2. *layout* : A dictionary/object controlling axes, camera, aspect ratio, margins, and titles.

Let's plot a 2D Gaussian probability density surface:$$f(x, y) = \frac{1}{2\pi} \exp\left(-\frac{x^2 + y^2}{2}\right)$$

If you are coming from Matplotlib `(fig, ax = plt.subplots())`, Plotly’s architecture feels very different at first because a Plotly figure is fundamentally a giant JSON (JavaScript Object Notation) data structure under the hood. Python simply builds this JSON dictionary and hands it off to your browser's JavaScript engine (WebGL/SVG) to draw.

In [2]:
import numpy as np
import plotly.graph_objects as go

#Create the 2D coordinate grid, standard numpy process
x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)

#Evaluate the scalar field Z = f(X, Y)
Z = (1 / (2 * np.pi)) * np.exp(-0.5 * (X**2 + Y**2))

#Initialize the Figure with a Surface trace
#in Plotly, go.Figure() creates the structured container object with two main properties:
#1.fig.data and 2. fig.layout (there is a 3rd that we will make when we get to animations)
fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, 
                                colorscale="Viridis", 
                                colorbar=dict(title="Density", thickness=15, len=0.7),
                                # Optional: project contours directly onto the z-floor, let us keep it since I plan on making them when we go to our library
                                contours=dict(
                                            z=dict(show=True, usecolormap=True, highlightcolor="white", project_z=True)
                                             ),
                               )
                     ]
               )

#Configure the 3D scene and layout
fig.update_layout(
    title="2D Standard Normal Density",
    width=750,
    height=600,
    margin=dict(l=10, r=10, t=50, b=10), #padding in pixel around the plot
    #we will come back to "scene" later 
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="f(x, y)",
               # 'cube' forces equal visual box proportions regardless of data scale
               aspectmode="cube",
               camera=dict( eye=dict(x=1.5, y=1.5, z=0.4)  # Initial viewing position
                          ),
              ),
)              

fig.show()

Note:

$\bullet$ Change the values inside `camera=dict(eye=dict(x=1.5, y=1.5, z=1.1))` to get the idea bout what they do.

$\bullet$ `go.surface` expects $2D$ arrays always. This is analogous to `fig, ax = plt.subplots()` whereas `fig.update_layout` is analogous to `ax.plot_surface(X, Y, Z, cmap='viridis')`.

$\bullet$ Why `fig.update_layout()`?? We never created a layout in the begining anyway. Actually we did, the previous `fig = go.Figure()` created a default figure. Explicitly put, we actually have two options:

In [3]:
fig = go.Figure(
                data=[go.Surface()],
                layout=go.Layout(title="2D Standard Normal Density", width=750)
               )

`layout` here is directly inside `go.Figure()` here. We choose not to do this. We do the following instead:

In [4]:
fig = go.Figure(data=[go.Surface()])
fig.update_layout(title="This creates an empty window",
                 width=750,
                 height=200,)

Here, we modify the layout separately. This is cleaner and keeps the data layer and plotting layer separate. The `fig.update_layout()` part is analogous to `ax.plot()` in Matplotlib. In 2D, axes are called "xaxis" and "yaxis". In 3D plots, the entire 3D coordinate is conatained inside a container called `scene` and it has several details such as titles, aspectmode and camera.

Now we see how to bring in interactive sliders. This is one of the most important things for the library we will be building. Recall, when we plotted 1D stochastic processes we saw the distribution of paths at several times $t_1, t_2, ..., t_n$. To do something similar in 3D, we will plot the paths in 3D with a slider that has several time points. Let us not go to processes and paths yet, for now, we focus on how to code the slider.

In modern Plotly (v6/v7), there are two ways to build a slider:

1. The Legacy `visible=True/False` Trace-Toggling Pattern. Adding n-many separate `go.Surface` traces to `fig.data` branch and toggling their visibility. For 3D surfaces, this bloats memory and lags badly in the browser.

2. The Modern `go.Frame` + `animate Pattern` (we will use this): We put one `go.Surface` in `fig.data`, store each parameter state as a lightweight `go.Frame` in `fig.frames`, and bind the slider to Plotly's hardware-accelerated "animate" engine. Recall how we had two branches `fig.data` and `fig.layout`. Now we will have the 3rd one named `fig.frames`, so it will be inside `go.Figure()`.

Let us apply this to the 2D Gaussian with various $\sigma$s.

In [5]:
import numpy as np
import plotly.graph_objects as go

x = np.linspace(-2, 2, 80)
y = np.linspace(-2, 2, 80)
X, Y = np.meshgrid(x, y)
sigmas = np.round(np.linspace(0.1, 1.5, 15), 2)

def gaussian_2d(sigma):
    return (1 / (2 * np.pi * sigma**2)) * np.exp(-0.5 * (X**2 + Y**2) / sigma**2)

#Create the initial figure (starting at sigmas[0] = 0.5)
fig = go.Figure(
            data=[go.Surface(x=X, y=Y, z=gaussian_2d(sigmas[0]), 
                             colorscale="Viridis", cmin=0, 
                             # cmax=gaussian_2d(sigmas[0]).max(),  
                            )
                 ],
            # Build one Frame per sigma value
            frames=[go.Frame(name=f"{s:.2f}",
                             data=[go.Surface(z=gaussian_2d(s))],  # Only update Z!
                            )
            for s in sigmas
                   ],
)

#Wire each frame name to a slider step
slider_steps = [dict(method="animate", label=f"{s:.2f}", 
                     args=[
                            [f"{s:.2f}"],     #must be wrapped in square brackets even for one frame
                            dict(mode="immediate", 
                                 frame=dict(duration=0, redraw=True)),
                          ],
                    )
                for s in sigmas
]

#Add the slider and lock the Z-axis range in layout
fig.update_layout(
    title="2D Gaussian Density: Varying Spread (σ)",
    width=750,
    height=650,
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="f(x, y)",
               zaxis=dict(range=[0, gaussian_2d(sigmas[0]).max()]),  # Prevent axis auto-rescaling
               aspectmode="cube",
               camera=dict(eye=dict(x=1.5, y=1.2, z=0.4)),
              ),
    sliders=[dict(active=0, 
                  currentvalue=dict(prefix="σ = ", font=dict(size=16)), 
                  pad=dict(t=40),
                  steps=slider_steps,
                 )
            ],
)

fig.show()

Note:

$\bullet$ By default, `go.Surface` maps the lowest value in the current Z matrix to the bottom of "Viridis" (dark purple) and the highest value in the current Z matrix to the top of "Viridis" (bright yellow). If we omit `cmax`, Plotly will still paint that tiny bump bright yellow! We have commented out `cmax` in our codes, you can include it and see that `cmax=gaussian_2d(sigmas[0]).max()` pins the colorbar scale globally so colors always correspond to true density height. I recommend you uncomment this and then move the slider to see what difference it makes.

$\bullet$ `name=f"{s:.2f}"`: Every frame must have a unique string name (e.g., "0.50", "0.70", ..., "2.50"). This string is the primary key that the slider uses to look up which snapshot to display. This naming is standard Python formatting inside a f-string `f"{}"`. Inside {} we have a separator denoted by "`:`" as usual and then on left we have the "Python_variable" and then on the right we have how we want it to be formatted `.2f` which means "round to 2 decimal places and in float format".

$\bullet$ Partial updates (`data=[go.Surface(z=)]`): Notice we did not pass x=X, y=Y, or colorscale="Viridis" inside go.Frame. In Plotly, a frame only needs to contain the diff (what changed relative to fig.data[0]). Plotly matches frames[i].data[0] to fig.data[0] by list index (0 $\rightarrow$ 0) and leaves x, y, and colorscale untouched.

$\bullet$ The most vital part is `slider_steps`. In Plotly, a slider doesn't automatically know about `fig.frames`. We have to build a list of step dictionaries, one per tick mark on the slider bar, that explicitly links each tick to a frame name. 

   1. `method="animate"`: Tells Plotly's JavaScript engine which internal action to run when this step is selected. (Other valid methods exist, like `"restyle"` or `"relayout"`, which edit properties directly without frames, whereas `"animate"` switches to a named go.Frame).
   2. `args=[]`: Every Plotly `method` takes a list of arguments (`args`). First element (`[f"{s:.2f}"]`): A list of frame names to play in sequence. Dragging a slider jumps to a single state, we pass a 1-element list containing just that frame's name and make sure this is warpped inside a square bracket. In the second element, we say how the animation will run. `mode="immediate"`  ensures that if we drag the slider fast across 5 ticks, cancel any queued intermediate frames and jump straight to where your mouse is right now; `duration=0` makes the transition feel like an instant snap; `redraw=True` is mandatory for 3D surfaces since by default we have (`redraw=False`) and Plotly only animates 2D SVG paths smoothly; 3D WebGL meshes (`go.Surface`) will refuse to update on screen unless `redraw=True` is explicitly set.

$\bullet$ We will finally attach the slider to `fig.update_layout()`. Note that we use `slider=[]` because we may have multiple sliders stacked below. `active=0` means the first state that it shows when it renders first is the $0^{th}$ state, that is, `sigma[0]`. `pad` ensure the plot and the sliders stay away from eachother.

These are very crucial for us since we will plot the distribution as 3D surfaces for paths soon and we want all these things, namely the slider, how to fix the height, smooth transition etc. In our library, we plan on showing the paths on the right and the distribution surfaces on the left. So, let us now see how to make two plots inside one window. In Matplotlib, we used $(1, 2)$ inside `subplots`, that is, `fig, (ax1, ax2) = plt.subplots(1, 2)` to create $1$ row and $2$ columns worth of space for plots. In Plotly, the exact equivalent is `make_subplots`, imported from `plotly.subplots`. However, there is one critical rule in Plotly when mixing 3D plots (`go.Surface`) and 2D plots (`go.Contour`, `go.Heatmap`, `go.Scatter`). Since Plotly uses different rendering engines for 2D (SVG/WebGL 2D) and 3D (WebGL 3D scene), we must tell `make_subplots` ahead of time which grid cells are 3D (`"scene"`) and which are 2D (`"xy"`) using the specs argument.

We will again create the Gaussian surface. We will have the surface itself on the left and the contour plot on the right.

In [6]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots #new import

x = np.linspace(-3, 3, 100)
y = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x, y)
Z = (1 / (2 * np.pi)) * np.exp(-0.5 * (X**2 + Y**2))

#initialize a 1-row, 2-column subplot grid with explicit subplot types
fig = make_subplots(rows=1, cols=2, 
                    specs=[[{"type": "scene"}, {"type": "xy"}]],  #left = 3D scene, right = 2D xy as explained before we have to specify
                    subplot_titles=("3D Surface View", "2D Top-Down Contour"),
                    horizontal_spacing=0.08,  #gap between the two panels (fraction of figure width)
                   )

#add the 3D Surface (called trace in Plotly) to row 1, col 1
#add_trace works same as what "data" branch was doing so far, if it helps we can always use this instead of data under the set-up of 1 row and 1 col
fig.add_trace(go.Surface(x=X, y=Y, z=Z, 
                         colorscale="Viridis", 
                         showscale=False,  # Hide duplicate colorbar so we only show one on the right
                        ),
              row=1, col=1,
             )

#add the 2D Contour plot to row 1, col 2
#here we will use go.Contour similar to go.Surface but this draws contours
fig.add_trace(go.Contour(x=x, y=y, #note that 2D Contour takes 1D x and y vectors
                         z=Z, 
                         colorscale="Viridis", 
                         contours=dict(coloring="heatmap", showlabels=True),  #smooth fill + numeric level labels, comment this out and see how the figure changes to figure out how this works
                         colorbar=dict(title="Density", thickness=15, len=0.8), 
                        ), 
              row=1, col=2,
             )

#style for both the 3D scene (left) and the 2D axes (right)
fig.update_layout(
    title="Joint 3D & 2D Views of Standard Normal Density",
    width=1050,
    height=520,
    margin=dict(l=20, r=20, t=80, b=30),
    # 3D panel settings (corresponds to row=1, col=1)
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="f(x, y)",
        aspectmode="cube",
        camera=dict(eye=dict(x=1.5, y=1.2, z=0.4)),
    ),
    # 2D panel settings (corresponds to row=1, col=2)
    #we dont need to do anything actuallu since contour are already created via go.Contour
    xaxis=dict(title="X", scaleanchor="y", scaleratio=1),  # Lock 1:1 aspect ratio so circles stay round!
    yaxis=dict(title="Y"),
)

fig.show()

Let us start with the easiest part. This is same as the Matplotlib's set up. I repeat this since when we get to paths these shapes will play a vital role in generating the plots. The "x", "y" are 1D arrays with 100 entries where as X has all x-coordinate entries of the grid created by x and y together containing 100*100 points; simialrly, Y has all the y-coordinates of the grid created by x and y together. So X (and Y) has the shape (100,100) like a 2D matrix. The Z is also (100, 100) shape since it calculates the value of the surface at `(X[i, j], Y[i, j])`. So it is important that we use (x,y,Z) for the Contour plot but (X,Y,Z) for the surface. 

We have `specs=[[{"type": "scene"}, {"type": "xy"}]]` because we wanted a 3D output on left and a 2D output in the right. If we wanted both to be 3D we will use `specs=[[{"type": "scene"}, {"type": "scene"}]]`. We will need that when we go to our library because, recall, we will have the 3D distribution surface on the left and the paths in 3D on right.

Note that when we have generated the `make_subplots` we use `fig = add_trace(trace, row= , col=)` to write down the details same as what data layer was doing so far. 

Setting `contours=dict(coloring="heatmap", showlabels=True)` gives you smooth color interpolation between level curves and prints the exact density values right on the contour rings. Comment this out to see the discrete plot it draws. Also try to comment out the `colorscale="Viridis"` with `contours=dict(coloring="heatmap", showlabels=True)` line intact to see the plot in heat map form without any "Viridis"-fication.
 
The code lines `xaxis=dict(scaleanchor="y", scaleratio=1)` is Plotly's exact equivalent of Matplotlib's `ax.set_aspect('equal')` since it forces 1 unit on the $x$-axis to equal 1 pixel-length unit on the $y$-axis. This keeps the contour lines circular.